# SLM-INDO · Notebook B: uji kelayakan TPU + pilot B0-S10

**Pengaturan Kaggle:** Accelerator = *TPU v5e-8*, Internet = *On*.
- **Bagian 1** (tanpa data): tes numerik GDN di TPU + `tpu_check.py` (kompilasi, throughput, MFU). Hasilnya di `tpu_check.json`.
- **Bagian 2** (butuh output Notebook A, *Add Input* → notebook A): pilot B0-S10 dan evaluasi bpb.

Tidak ada angka di sini yang menjadi hasil paper sebelum protokol pilot selesai (KONSEP §10D).

In [ ]:
import subprocess, os
def sh(cmd):
    """Jalankan perintah shell, tampilkan output langsung, hentikan notebook bila gagal."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"perintah gagal: {cmd}")

In [ ]:
sh("pip install -q optax pytest")
sh("python -c 'import jax, optax; print(jax.__version__, optax.__version__); print(jax.devices())'")
sh("rm -rf /tmp/SLM-INDO && git clone -q https://github.com/rexxar280903/SLM-INDO.git /tmp/SLM-INDO")
sh("cd /tmp/SLM-INDO && git log --oneline -1 | tee /kaggle/working/code_commit.txt")
os.chdir("/tmp/SLM-INDO")

In [ ]:
# Bagian 1a: matematika GDN di TPU (chunkwise = rekuren)
sh("python -m pytest -q tests/test_gdn.py")

In [ ]:
# Bagian 1b: uji kelayakan TPU (Keputusan B)
sh("python scripts/tpu_check.py --out /kaggle/working/tpu_check.json --steps 20")

In [ ]:
# Bagian 2: pilot B0-S10 (lewati bila data Notebook A belum ditambahkan sebagai input)
import glob
cands = glob.glob("/kaggle/input/**/tokens16k/train.bin", recursive=True)
DATA = os.path.dirname(cands[0]) if cands else None
EVAL = os.path.join(os.path.dirname(DATA), "eval") if DATA else None
print("data:", DATA)
if DATA:
    sh(f"PYTHONPATH=src python -m slm.train --arch b0 --scale s10 --seed 0 --data-dir {DATA} --out-dir /kaggle/working/runs")
    sh(f"PYTHONPATH=src python -m slm.evaluate --run-dir /kaggle/working/runs/b0-s10-seed0-flops --data-dir {DATA} --eval-dir {EVAL} --sets val flores_dev --batch 64")